# Fine-tuning Gemma 4 E2B for ABSA (Colab, QLoRA via Unsloth)

Trains a LoRA adapter on the SemEval restaurant training split, then exports a Q4_K_M GGUF that runs
in Ollama through the same pipeline as the zero-/few-shot models. Run on a **T4 GPU** runtime
(Runtime -> Change runtime type -> T4 GPU).

Steps: mount Drive -> install Unsloth -> build instruction data from `train.json` -> QLoRA train ->
quick sanity check -> export GGUF (Q4_K_M) to Drive. Local registration in Ollama (`ollama create`)
is done afterwards on your own machine; the resulting model name is then added to the main notebook.

The training prompt is the **same** zero-shot prompt the main notebook uses, so the adapter learns to
answer exactly the input it will be served at inference time.

## 1. Runtime check and Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Set Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

# adjust to wherever you keep the project on Drive
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/absa")     # <-- change if needed
DATA_DIR = PROJECT / "data"                         # expects train.json here
OUT_DIR  = PROJECT / "finetune" / "gemma4_e2b"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("train.json present:", (DATA_DIR / "train.json").exists())

## 2. Install Unsloth

Pinning is deliberate: Gemma 4 is new, so an up-to-date Unsloth is required, but a known merge
regression on Gemma 4 (below) means you want a version you can reproduce. If install resolves a
broken combo, restart the runtime once and re-run.

In [ ]:
%%capture
!pip install -U unsloth unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import unsloth, transformers, trl
print("unsloth", unsloth.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__)

## 3. Build instruction data from the split

Each example is `(prompt, completion)`: the prompt is the notebook's zero-shot prompt for the
sentence, the completion is the gold aspect/sentiment list as compact JSON. Sentences with no aspect
term map to `[]`, which teaches the model to abstain rather than hallucinate aspects.

`SYSTEM_INSTRUCTION` / `build_prompt` are copied verbatim from the main notebook so the formats can't
drift apart. If you change the prompt there, change it here too.

In [ ]:
import json
SENTIMENTS = ["positive", "negative", "neutral", "conflict"]

# --- copied verbatim from the main notebook (keep in sync) ---
SYSTEM_INSTRUCTION = (
    "You are an aspect-based sentiment analysis system for customer reviews.\n"
    "Given one sentence, extract every aspect term that is explicitly mentioned and its sentiment.\n"
    "An aspect term is a word or phrase copied verbatim from the sentence that names something being "
    "evaluated.\n"
    "sentiment is exactly one of: positive, negative, neutral, conflict "
    "(conflict = both positive and negative).\n"
    "If the sentence contains no aspect term, return an empty array.\n"
    "Return ONLY a JSON array and nothing else. "
    'Each element must be {"aspect": "<term>", "sentiment": "<sentiment>"}.'
)

def build_prompt(text, examples=None):
    parts = [SYSTEM_INSTRUCTION, ""]
    if examples:
        for ex in examples:
            parts.append(f'Sentence: "{ex["text"]}"')
            parts.append("Output: " + json.dumps(
                [{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in ex["aspects"]],
                ensure_ascii=False))
            parts.append("")
    parts.append(f'Sentence: "{text}"')
    parts.append("Output:")
    return "\n".join(parts)
# --- end copied section ---

train = json.loads((DATA_DIR / "train.json").read_text(encoding="utf-8"))
print("train sentences:", len(train))

def to_completion(s):
    return json.dumps([{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in s["aspects"]],
                      ensure_ascii=False)

rows = [{"prompt": build_prompt(s["text"]), "completion": to_completion(s)} for s in train]
print("example:\n", rows[0]["prompt"][:200], "\n-> ", rows[0]["completion"])

## 4. Load Gemma 4 E2B (4-bit) and attach LoRA

QLoRA: the base model loads in 4-bit, only the LoRA adapters train. LoRA targets attention and MLP
projections, rank 16 (good default for a format+domain adaptation task this size).

In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ = 1024  # ABSA inputs are short; small value saves VRAM and speeds training

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = MAX_SEQ,
    dtype = None,            # auto
    load_in_4bit = True,     # QLoRA
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 5. Format and tokenize

The model trains on the full `prompt + completion + EOS` string but the loss is computed on the
completion only (train_on_responses), so it learns to *produce* the JSON, not to echo the prompt. The
response marker must match the prompt: completions begin right after `Output:`.

In [ ]:
from datasets import Dataset

EOS = tokenizer.eos_token
def to_text(r):
    return {"text": r["prompt"] + " " + r["completion"] + EOS}

ds = Dataset.from_list(rows).map(to_text)
# small holdout to watch for overfitting (not a real test set; that lives in the main notebook)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(ds)
print(ds["train"][0]["text"][:300])

## 6. Train (3 epochs)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["test"],
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,     # effective batch 8
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = str(OUT_DIR / "checkpoints"),
        report_to = "none",
    ),
)

# compute loss on the answer only; the second marker is what the prompt ends with
trainer = train_on_responses_only(
    trainer,
    instruction_part = "Sentence:",
    response_part = "Output:",
)

stats = trainer.train()
print(stats)

A training loss settling around **13-15 is normal for Gemma 4 E2B/E4B** (a known quirk of these
multimodal models), not a bug. Losses in the hundreds would instead point to a gradient-accumulation
problem. Watch the eval loss across the 3 epochs: if it starts rising while train loss keeps falling,
that's overfitting and 2 epochs would be the safer choice on a re-run.

## 7. Quick sanity check

In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)

sample = train[0]["text"]
inputs = tokenizer(build_prompt(sample), return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False)
print("INPUT :", sample)
print("OUTPUT:", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 8. Export to GGUF (Q4_K_M)

One call does merge + convert + quantize. Q4_K_M matches the quantization of the zero-/few-shot
models so the only thing that differs at evaluation is the fine-tuning itself.

**Known Gemma 4 pitfall:** the merge step has been reported to overwrite the chat `eos_token`,
which makes the exported model generate without stopping and run far slower. The cell after the
export checks the saved `eos_token` and warns if it looks wrong, so you catch it here and not after
a confusing slow run in Ollama.

In [ ]:
GGUF_DIR = OUT_DIR / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method = "q4_k_m",
)
print("saved to", GGUF_DIR)
!ls -lh "{GGUF_DIR}"

In [ ]:
# verify the chat eos_token survived the merge (Gemma 4 regression guard)
import json, glob
cfgs = glob.glob(str(GGUF_DIR / "**" / "tokenizer_config.json"), recursive=True)
if cfgs:
    cfg = json.loads(open(cfgs[0]).read())
    eos = cfg.get("eos_token")
    print("eos_token:", eos)
    if eos in ("<eos>", "</s>"):
        print("WARNING: eos_token looks like the base eos, not the chat end-of-turn token.")
        print("If the model won't stop generating in Ollama, set the stop token in the Modelfile.")
else:
    print("no tokenizer_config.json found under GGUF dir; check export output above")

## 9. Next (local machine, not Colab)

1. Download the `.gguf` from Drive (`finetune/gemma4_e2b/gguf/`).
2. Write a `Modelfile`:
   ```
   FROM ./gemma4-e2b-absa-Q4_K_M.gguf
   PARAMETER temperature 0
   PARAMETER stop "<end_of_turn>"
   ```
   (Unsloth often writes a Modelfile with the correct template alongside the GGUF; prefer that one if
   present.)
3. Register it: `ollama create gemma4-e2b-absa-ft -f Modelfile`
4. In the main notebook add the name to a fine-tuned models list and run it through the same `run()`
   loop with `mode="finetuned"`. Metrics, caching and plots already handle it.